# Simulated 2-qubit scheduling-policy comparison

This notebook mirrors `scheduling_comparison.ipynb`, but it keeps the QUBEX experiment in `mock_mode=True` and does not connect to hardware. It uses calibration/config files matching the selected physical qubit labels to build the same `device-topology`, then compares Qiskit ALAP/ASAP scheduling with the legacy Device Gateway timing policy at the pulse-schedule level.

The population-dynamics cell is a pulse-level qxsimulator run. It is not guaranteed to be meaningful just because the real-device `config`, `params`, and `calibration/calib_note.json` are present: the pulse amplitudes, frequencies, channel targets, coupling model, and units must be calibrated for the simulator model. Treat that cell as a simulation-calibration check, not as a validation that raw hardware calibration reproduces ideal circuit dynamics.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import SVG, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qubex import Experiment
from qxsimulator import QuantumSimulator

# Re-running this cell in an existing notebook kernel should pick up local provider edits.
for module_name in (
    "qiskit_qubex_provider.dynamical_decoupling",
    "qiskit_qubex_provider.executor",
    "qiskit_qubex_provider.backend",
    "qiskit_qubex_provider.provider",
    "qiskit_qubex_provider",
):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

from qiskit_qubex_provider import (
    QubexProvider,
    build_pulse_schedule_timeline_figure,
    build_qxsimulator_system,
    diff_pulse_schedules,
    filter_pulse_schedule_for_simulation,
    summarize_pulse_schedule,
)

HERE = Path.cwd()
if not (HERE / "bell_state.py").exists():
    HERE = Path("examples/hardware").resolve()

spec = importlib.util.spec_from_file_location("hardware_bell_state", HERE / "bell_state.py")
bell_state = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(bell_state)


In [ ]:
DEVICE_ID = bell_state.DEFAULT_DEVICE_ID
QUBIT_LABELS = bell_state.DEFAULT_QUBIT_LABELS
BELL_PAIR = bell_state.DEFAULT_BELL_PAIR
WORKLOAD_QUBIT_LABELS = BELL_PAIR
CONFIG_ROOT = HERE / "qubex-config"
OUTPUT_DIR = HERE / "generated"

NUM_QUBITS = len(WORKLOAD_QUBIT_LABELS)
SIMULATION_SHOTS = 4096
PULSE_SIMULATION_DT_NS = 4.0
PULSE_SIMULATION_SAMPLES = 250
CALIBRATION_VALID_DAYS = 10000
N_STEPS = 1
TOTAL_TIMES = np.linspace(0, 5, 30)
TOTAL_TIMES_IDEAL = np.linspace(0, 5, 241)
SAMPLE_TOTAL_TIME = 2.5
REFERENCE_J_VALUES = np.array([0.542641286533492])

TOPOLOGY_PATH = bell_state.topology_json_path(
    output_dir=OUTPUT_DIR,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
)

device_config = CONFIG_ROOT / DEVICE_ID
required_calibration_paths = (
    device_config / "config",
    device_config / "params",
    device_config / "calibration" / "calib_note.json",
)
missing_calibration_paths = [path for path in required_calibration_paths if not path.exists()]
if missing_calibration_paths:
    missing = "\n".join(f"  - {path}" for path in missing_calibration_paths)
    raise FileNotFoundError(
        "Pulse schedule generation requires a QUBEX calibration/config tree "
        "matching DEVICE_ID and QUBIT_LABELS. Meaningful qxsimulator "
        "population dynamics additionally require simulator-compatible "
        "pulse calibration; raw hardware calibration may not be enough. "
        "Missing:\n"
        f"{missing}"
    )

mock_experiment = Experiment(
    chip_id=DEVICE_ID,
    qubits=QUBIT_LABELS,
    config_dir=device_config / "config",
    params_dir=device_config / "params",
    calib_note_path=device_config / "calibration" / "calib_note.json",
    mock_mode=True,
)

bell_state.generate_device_topology(
    config_root=CONFIG_ROOT,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
    bell_pair=BELL_PAIR,
    output_path=TOPOLOGY_PATH,
    pulse_source=mock_experiment,
    calibration_valid_days=CALIBRATION_VALID_DAYS,
)


def provider_from_mock_experiment(*, timing_policy: str) -> QubexProvider:
    return QubexProvider.from_experiment(
        mock_experiment,
        name=f"{DEVICE_ID}-{timing_policy}-simulation",
        device_topology=TOPOLOGY_PATH,
        qubit_labels=QUBIT_LABELS,
        timing_policy=timing_policy,
        native=True,
        calibration_valid_days=CALIBRATION_VALID_DAYS,
        execute_options={
            "state_classification": False,
            "time_integration": True,
            "plot": False,
        },
    )


provider_qiskit = provider_from_mock_experiment(timing_policy="qiskit")
provider_legacy = provider_from_mock_experiment(timing_policy="legacy_device_gateway")
backend_qiskit = provider_qiskit.get_backend()
backend_legacy = provider_legacy.get_backend()
backend_ideal = QubexProvider(num_qubits=NUM_QUBITS, coupling_map=[(0, 1)]).get_backend()
initial_layout = bell_state.bell_initial_layout(
    qubit_labels=QUBIT_LABELS,
    bell_pair=BELL_PAIR,
)

print("qiskit backend:", backend_qiskit.name)
print("legacy backend:", backend_legacy.name)
print("topology:", TOPOLOGY_PATH)
print("topology svg:", TOPOLOGY_PATH.with_suffix(".svg"))
print("experiment qubits:", QUBIT_LABELS)
print("logical Heisenberg qubits:", WORKLOAD_QUBIT_LABELS)
print("initial layout:", initial_layout)
print("J values:", REFERENCE_J_VALUES)


In [ ]:
display(SVG(filename=str(TOPOLOGY_PATH.with_suffix(".svg"))))


In [ ]:
def create_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def uncompute_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def add_xz_pauli_rotation(qc: QuantumCircuit, q0: int, q1: int, angle: float) -> None:
    qc.h(q0)
    qc.rzz(angle, q0, q1)
    qc.h(q0)


def add_heisenberg_interactions(
    qc: QuantumCircuit,
    edge: tuple[int, int],
    coupling: float,
    evolution_time: float,
) -> None:
    control, target = edge
    theta = float(coupling) * evolution_time
    qc.cx(control, target)
    qc.rx(2.0 * theta, control)
    qc.rz(2.0 * theta, target)
    add_xz_pauli_rotation(qc, control, target, -2.0 * theta)
    qc.cx(control, target)


HEISENBERG_EDGE = (0, 1)


def heisenberg_2q_circuit(total_time: float, *, n_steps: int = N_STEPS) -> QuantumCircuit:
    qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
    create_half_ghz_state(qc)

    step_time = float(total_time) / n_steps
    for _ in range(n_steps):
        add_heisenberg_interactions(qc, HEISENBERG_EDGE, REFERENCE_J_VALUES[0], step_time)

    uncompute_half_ghz_state(qc)
    qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))
    return qc


def probability_zero_from_counts(counts: dict[str, int]) -> float:
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return counts.get("0" * NUM_QUBITS, 0) / total


def exact_probability_zero(circuit: QuantumCircuit) -> float:
    state_circuit = circuit.remove_final_measurements(inplace=False)
    probabilities = Statevector.from_instruction(state_circuit).probabilities_dict()
    return float(probabilities.get("0" * NUM_QUBITS, 0.0))


def compile_physical(total_time: float, backend) -> QuantumCircuit:
    return transpile(
        heisenberg_2q_circuit(total_time),
        backend,
        initial_layout=initial_layout,
        optimization_level=1,
        seed_transpiler=7,
    )


def compile_qiskit(
    total_time: float,
    *,
    scheduling_method: str = "alap",
) -> QuantumCircuit:
    physical = compile_physical(total_time, backend_qiskit)
    return transpile(
        physical,
        backend_qiskit,
        scheduling_method=scheduling_method,
        optimization_level=0,
    )


def compile_legacy(total_time: float) -> QuantumCircuit:
    return compile_physical(total_time, backend_legacy)


def compile_pulse_simulation_schedule(
    total_time: float,
    *,
    scheduling_method: str = "alap",
):
    circuit = heisenberg_2q_circuit(total_time).remove_final_measurements(inplace=False)
    physical = transpile(
        circuit,
        backend_qiskit,
        initial_layout=initial_layout,
        optimization_level=1,
        seed_transpiler=7,
    )
    scheduled = transpile(
        physical,
        backend_qiskit,
        scheduling_method=scheduling_method,
        optimization_level=0,
    )
    return filter_pulse_schedule_for_simulation(backend_qiskit.validate(scheduled)[0])


heisenberg_2q_circuit(SAMPLE_TOTAL_TIME).draw("text")


In [ ]:
def simulate_counts(total_time: float) -> dict[str, int]:
    circuit = transpile(
        heisenberg_2q_circuit(total_time),
        backend_ideal,
        optimization_level=1,
        seed_transpiler=7,
    )
    return backend_ideal.run(circuit, shots=SIMULATION_SHOTS).result().get_counts()


ideal = [
    exact_probability_zero(heisenberg_2q_circuit(float(total_time)))
    for total_time in TOTAL_TIMES_IDEAL
]
shot_simulation = [
    probability_zero_from_counts(simulate_counts(float(total_time)))
    for total_time in TOTAL_TIMES
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal,
    name="statevector reference",
    line=dict(color="#475569", dash="dot"),
))
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES,
    y=shot_simulation,
    name=f"ideal shot simulator ({SIMULATION_SHOTS} shots)",
    mode="lines+markers",
    line=dict(color="#2563eb"),
))
fig.update_layout(
    title="2-qubit Heisenberg ideal simulation",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="probability of |00>", range=[-0.05, 1.05]),
    width=760,
    height=430,
)
fig.show()

print("statevector min:", float(np.min(ideal)))
print("shot simulation min:", float(np.min(shot_simulation)))


In [ ]:
QISKIT_COMPILE_VARIANTS = {
    "qiskit ALAP": dict(scheduling_method="alap"),
    "qiskit ASAP": dict(scheduling_method="asap"),
}

sample_circuits = {
    name: compile_qiskit(SAMPLE_TOTAL_TIME, **options)
    for name, options in QISKIT_COMPILE_VARIANTS.items()
}
sample_circuits["legacy_device_gateway"] = compile_legacy(SAMPLE_TOTAL_TIME)

sample_schedules = {
    name: backend_qiskit.validate(circuit)[0]
    for name, circuit in sample_circuits.items()
    if name.startswith("qiskit")
}
sample_schedules["legacy_device_gateway"] = backend_legacy.validate(
    sample_circuits["legacy_device_gateway"]
)[0]

for name, schedule in sample_schedules.items():
    print(f"{name}")
    print(summarize_pulse_schedule(schedule))

print()
print("qiskit ALAP vs ASAP diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["qiskit ASAP"]))

print()
print("qiskit ALAP vs legacy diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["legacy_device_gateway"]))


In [ ]:
for name, schedule in sample_schedules.items():
    build_pulse_schedule_timeline_figure(
        schedule,
        title=f"2q Heisenberg sample: {name}",
    ).show()


In [ ]:
qx_system = build_qxsimulator_system(
    TOPOLOGY_PATH,
    qubit_labels=WORKLOAD_QUBIT_LABELS,
    qubit_dimension=2,
    include_decoherence=False,
)
pulse_schedule = compile_pulse_simulation_schedule(
    SAMPLE_TOTAL_TIME,
    scheduling_method="alap",
)
print("qxsimulator objects:", qx_system.object_labels)
print("pulse simulation channels:", pulse_schedule.labels)
print("channel frequencies:", pulse_schedule.get_frequencies())
print("channel targets:", pulse_schedule.get_targets())

pulse_result = QuantumSimulator(qx_system).mesolve(
    pulse_schedule,
    dt=PULSE_SIMULATION_DT_NS,
    n_samples=PULSE_SIMULATION_SAMPLES,
)
pulse_result.plot_population_dynamics(n_samples=PULSE_SIMULATION_SAMPLES)


In [ ]:
def final_population(result, basis: str) -> float:
    populations = np.real(result.final_state.diag())
    return float(populations[result.system.basis_labels.index(basis)])


ideal_p00 = exact_probability_zero(heisenberg_2q_circuit(SAMPLE_TOTAL_TIME))
pulse_p00 = final_population(pulse_result, "00")
print("ideal final P(|00>):", ideal_p00)
print("pulse final P(|00>):", pulse_p00)
print("absolute difference:", abs(pulse_p00 - ideal_p00))


In [ ]:
duration_sweep = {}
for name, options in QISKIT_COMPILE_VARIANTS.items():
    duration_sweep[name] = [
        backend_qiskit.validate(compile_qiskit(float(total_time), **options))[0].duration
        for total_time in TOTAL_TIMES
    ]
duration_sweep["legacy_device_gateway"] = [
    backend_legacy.validate(compile_legacy(float(total_time)))[0].duration
    for total_time in TOTAL_TIMES
]

fig = go.Figure()
for name, durations in duration_sweep.items():
    fig.add_trace(go.Scatter(
        x=TOTAL_TIMES,
        y=durations,
        mode="lines+markers",
        name=name,
    ))
fig.update_layout(
    title="Pulse-schedule duration by timing policy",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="schedule duration [ns]"),
    width=760,
    height=430,
)
fig.show()
